# Notebook para geração de pasta para o yousub - Modelo de Coral Sol - Segmentação
Autora: Viviane da Rosa Sommer

Entrega: 10/04/2025
## Objetivo:
Notebook para realizar inferência em um determinado vídeo e gerar a pasta consumida pela interface do YouSub


## Instalações e Importações Necessárias

In [1]:
import glob
import math
import os
import time
import json
from datetime import datetime, timedelta
import ffmpeg
import re
import cv2
import numpy as np
import pandas as pd
import keras
import matplotlib.pyplot as plt
from keras.preprocessing.image import load_img, img_to_array
from utils.predict_functions import get_dataframe_class
from tqdm import tqdm
import subprocess
from ultralytics import YOLO
import torch
import torchvision
import tempfile
import mimetypes
import boto3
from dotenv import load_dotenv

2025-04-14 13:40:30.898840: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-14 13:40:30.910495: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-14 13:40:30.914080: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-14 13:40:30.923394: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Declaração de Constantes 

#### PATH's dos vídeos e modelos

In [2]:
bucket_name = 'analise-dados'
model_key= 'projeto-ia-submarina/ia-frente-ambiental/Models/coral/segm/coral_yolov11_segm_4c_V5.pt'
result_path = 'input_yousub'
video_keys = [
    'projeto-ia-submarina/ia-yousub/NOAA-yousub/videos_entrada_NOAA/07-OS006000761716-STRMLZ25-183/VIDEOS/HD/STRMLZ25-183-OS006000761716-2025-04-03-T-17-15-08-D30.MP4',
    'projeto-ia-submarina/ia-yousub/NOAA-yousub/videos_entrada_NOAA/07-OS006000761716-STRMLZ25-183/VIDEOS/HD/STRMLZ25-183-OS006000761716-2025-04-03-T-17-45-09-D30.MP4',
    'projeto-ia-submarina/ia-yousub/NOAA-yousub/videos_entrada_NOAA/07-OS006000761716-STRMLZ25-183/VIDEOS/HD/STRMLZ25-183-OS006000761716-2025-04-03-T-17-52-33-D07.MP4',
    'projeto-ia-submarina/ia-yousub/NOAA-yousub/videos_entrada_NOAA/07-OS006000761716-STRMLZ25-183/VIDEOS/HD/STRMLZ25-183-OS006000761716-2025-04-03-T-18-22-35-D30.MP4',
    'projeto-ia-submarina/ia-yousub/NOAA-yousub/videos_entrada_NOAA/07-OS006000761716-STRMLZ25-183/VIDEOS/HD/STRMLZ25-183-OS006000761716-2025-04-03-T-18-27-19-D04.MP4'
]

predicoes_path = os.path.join(result_path, 'predicoes')
missing_timestamps = {}
CORAL_CLASS_ID = 0
columns = ['Video', 'Frame', 'Tempo', 'Timestamp']
videos = []

name_os = '07-OS006000761716-STRMLZ25-183'
local_folder_path = 'input_yousub'
s3_folder_path = 'projeto-ia-submarina/ia-yousub/NOAA-yousub/output_yousub_NOAA/' + 'yousub_input_' + name_os

#### Configurar boto3 com credenciais, para acessar o S3.
 O arquivo deve-se chamar config.json e deve ter o seguinte formato:
 [
  {
    "aws_access_id": "SEU_AWS_ACCESS_KEY_ID",
    "secret_key": "SUA_AWS_SECRET_ACCESS_KEY"
  }
]

In [3]:
with open("config.json",'r') as f:
    config = json.load(f)

data_config = config[0]
s3 = boto3.client( 's3', aws_access_key_id = data_config["aws_access_id"], aws_secret_access_key = data_config["secret_key"])

## Declaração de funções

### Funções específicas do YouSub

In [4]:
def build_sprite(miniatures, save_path, sprite_width=1920, sprite_height=1080, columns=10, lines=10):
    individual_width = sprite_width // columns
    individual_height = sprite_height // lines
    sprite = np.zeros((sprite_height, sprite_width, 3), dtype=np.uint8)
    for idx, miniature in enumerate(miniatures):
        if idx >= columns * lines:
            break
        if miniature.shape[1] != individual_width or miniature.shape[0] != individual_height:
            print(f"Image {miniature} has dimensions different from {individual_width}x{individual_height}")
            continue
        line = idx // columns
        col = idx % columns
        y_start = line * individual_height
        y_end = y_start + individual_height
        x_start = col * individual_width
        x_end = x_start + individual_width
        sprite[y_start:y_end, x_start:x_end] = miniature
    cv2.imwrite(save_path, sprite)
    print(f'Final sprite saved at {save_path}')

def extract_miniatures(video, num_miniatures=100, dims=(192, 108)):
    cap = cv2.VideoCapture(video)
    num_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    frame_skip = num_frames // num_miniatures
    miniatures = []
    for frame_count in range(num_miniatures):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_skip * frame_count)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, dims, interpolation=cv2.INTER_LINEAR)
        miniatures.append(frame)
    cap.release()
    return miniatures

def get_timestamp(file):
    # Pattern to match the timestamp format in the filename
    pattern = r"(\d{4}-\d{2}-\d{2})-T-(\d{2})-(\d{2})-(\d{2})"
    format = "%Y-%m-%d %H:%M:%S"
    
    # Try to extract timestamp from filename
    result = re.search(pattern, file)
    if result:
        # Extract the date and time components from the filename
        date = result.group(1)
        hour = result.group(2)
        minute = result.group(3)
        second = result.group(4)
        # Create a formatted timestamp
        formatted_timestamp = f"{date} {hour}:{minute}:{second}"
        return datetime.strptime(formatted_timestamp, format), True  
    else:
        # If no timestamp is found in the video title, generate the timestamp using datetime.now
        current_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        missing_timestamps[file] = datetime.strptime(current_timestamp, format)
        return datetime.strptime(current_timestamp, format), False  


# Functions to extract video data
def extract_timestamp_from_filename(filename, missing_timestamps):
    # Regular expression to extract timestamp from video title
    match = re.search(r"(\d{4}-\d{2}-\d{2})-T-(\d{2}-\d{2}-\d{2})", filename)
    if match:
        date_str = match.group(1)  # 2024-05-12
        time_str = match.group(2)  # 09-52-54
        # Combine datetime parts
        timestamp_str = f"{date_str} {time_str.replace('-', ':')}"
        # Convvert to a datetime object
        timestamp_inicial = datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S")
        return timestamp_inicial
    else:
        return missing_timestamps[filename]

def calculate_final_timestamp(timestamp_inicial, duracao_segundos):
    # Estima o timestamp final somando a duração ao timestamp inicial
    timestamp_final = timestamp_inicial + timedelta(seconds=duracao_segundos)
    return timestamp_final

def get_video_duration(video_path):
    # Full path to ffprobe in the current environment
    ffprobe_path = 'ffmpeg-7.0.2-amd64-static/ffprobe'

    # Command to run ffprobe and get the duration
    cmd = [
        ffprobe_path, 
        '-v', 'error', 
        '-select_streams', 'v:0', 
        '-show_entries', 'stream=duration', 
        '-of', 'default=noprint_wrappers=1:nokey=1', 
        video_path
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    if result.returncode != 0:
        raise Exception(f"Error with ffprobe: {result.stderr}")

    # Duração em segundos extraída do ffprobe
    return math.floor(float(result.stdout.strip()))

### Funções específicas do modelo - Coral Sol

In [5]:
def upload_folder_to_s3(local_folder_path, bucket_name, s3_folder_path, video_keys):
    """
    Uploads a local folder and its contents, including multiple videos, to an S3 bucket.

    Args:
        local_folder_path (str): The path to the local folder.
        bucket_name (str): The name of the S3 bucket.
        s3_folder_path (str): The S3 key prefix where the folder will be uploaded.
        video_keys (list): A list of S3 keys for the videos to download and upload.
    """
    try:
        # Upload folder files
        for root, dirs, files in os.walk(local_folder_path):
            for file in files:
                local_file_path = os.path.join(root, file)
                relative_path = os.path.relpath(local_file_path, local_folder_path)
                s3_file_key = os.path.join(s3_folder_path, relative_path)
                content_type, _ = mimetypes.guess_type(local_file_path)
                extra_args = {'ContentType': content_type} if content_type else {}
                try:
                    s3.upload_file(local_file_path, bucket_name, s3_file_key, ExtraArgs=extra_args)
                    print(f"Uploaded {local_file_path} to s3://{bucket_name}/{s3_file_key}")
                except Exception as upload_error:
                    print(f"Error uploading {local_file_path}: {upload_error}")
                    return  # Return early if a file fails to upload.

        print(f"Folder '{local_folder_path}' uploaded to s3://{bucket_name}/{s3_folder_path}")
    except Exception as e:
        print(f"Error walking through folder: {e}")


def load_model_from_s3(bucket_name, model_key):
    """Loads an Ultralytics YOLO model from S3 into memory."""
    class_list = ['CoralSol','notCoralSol']  # model classes
    try:
        # Download the model to a temporary file
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as temp_file:
            s3.download_fileobj(bucket_name, model_key, temp_file)
            temp_file_path = temp_file.name

        model = YOLO(temp_file_path)  # Load the YOLO model from the downloaded file.
        os.unlink(temp_file_path) # Delete the temp file.
        return model, class_list
    except Exception as e:
        print(f"Error loading model from S3: {e}")
        return None, None  # Return None for both model and class_list in case of error


def crop_frame(frame: np.ndarray) -> tuple:
    """
    Crops the frame to focus on the central region.

    Parameters:
        frame (np.ndarray): The frame to crop.

    Returns:
        tuple: A tuple containing the cropped frame and the top and bottom crop coordinates.
    """
    height, _, _ = frame.shape
    mid = height // 2
    top = max(0, mid - int(0.34 * height))
    bottom = min(height, mid + int(0.34 * height))
    cropped_frame = frame[top:bottom, :]
    return cropped_frame, top, bottom

def process_results(results: list) -> any:
    """
    Processes the results obtained from the model, returning the first valid result with masks.

    Parameters:
        results (list): List of model detection results.

    Returns:
        any: The first valid result containing masks, or None if no valid result is found.
    """
    for result in results:
        if hasattr(result, 'masks') and result.masks is not None:
            return result
    return None

def prediction_coral(coral_model, img: np.ndarray) -> tuple:
    """
    Predicts coral objects using the coral model and returns individual masks and their scores.

    Parameters:
        img (np.ndarray): Input image for prediction.

    Returns:
        tuple: A tuple containing:
            - masks (list): List of binary masks for the predicted coral regions.
            - scores (list): List of scores for each predicted coral region.
    """
    coral_results = coral_model(img, verbose=False)
    coral_best_result = process_results(coral_results)

    masks_list = []
    scores_list = []

    if coral_best_result is not None and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        if len(coral_indices) > 0:
            coral_boxes = boxes[coral_indices]
            coral_masks = masks[coral_indices]
            coral_scores = scores[coral_indices]

            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            for idx in nms_indices:
                mask_np = coral_masks[idx].int().cpu().numpy() * 255
                masks_list.append(mask_np.astype(np.uint8))
                scores_list.append(coral_scores[idx].item())

    return masks_list, scores_list

def overlay_masks(frame: np.ndarray, masks_list: list, top: int, bottom: int, scores_list: list) -> np.ndarray:
    """
    Overlays the masks on the original frame and adds annotations.

    Parameters:
        frame (np.ndarray): The original video frame.
        masks_list (list): List of binary masks to overlay.
        top (int): The top crop coordinate.
        bottom (int): The bottom crop coordinate.
        scores_list (list): List of scores for each mask.

    Returns:
        np.ndarray: The frame with the masks overlaid and annotations.
    """
    if not masks_list:
        center_y = frame.shape[0] // 2
        cv2.putText(frame, "No corals detected", (10, center_y), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        return frame

    for mask, score in zip(masks_list, scores_list):
        mask_resized = cv2.resize(mask, (frame.shape[1], bottom - top))
        contours, _ = cv2.findContours(mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(frame[top:bottom, :], contours, -1, (0, 0, 255), 2)
        for contour in contours:
            x, y, _, _ = cv2.boundingRect(contour)
            score_text = f'{score:.2f}'
            cv2.putText(frame, score_text, (x, y + top - 5), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 1, cv2.LINE_AA)

    return frame

def download_video_from_s3(bucket_name, video_key):
    """
    Downloads a video from S3 and returns the path to the downloaded file.

    Args:
        bucket_name (str): The name of the S3 bucket.
        video_key (str): The S3 key of the video.

    Returns:
        str or None: The path to the downloaded video file, or None if an error occurs.
    """
    try:
        # Download the video to a temporary file
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp4") as temp_file:
            s3.download_fileobj(bucket_name, video_key, temp_file)
            temp_file_path = temp_file.name #get the path to the temporary file.
            return temp_file_path #return the path, not the file object.
    except Exception as e:
        print(f"Error downloading video {video_key}: {e}")
        return None


def process_video(model, video, total_videos, total_inferences, total_frames, class_list, current_timestamp = None):
    df_inference = []
    #model_shape = model.layers[0].get_output_at(0).get_shape().as_list()

    cap = cv2.VideoCapture(video)
    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    frame_skip = int(frame_rate)
    num_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    print(os.path.basename(video))
    video_name = os.path.basename(video).split('.')[0]
    initial_timestamp = current_timestamp
    if initial_timestamp is None:
        initial_timestamp, _ = get_timestamp(video_name)
    else:
        missing_timestamps[video_name] = initial_timestamp
        print(missing_timestamps)
        

    # Create a tqdm progress bar for the loop
    with tqdm(total=int(num_frames // frame_skip), desc=f"Processing {video_name}") as pbar:
        for prediction_index in range(0, int(num_frames), frame_skip):
            cap.set(cv2.CAP_PROP_POS_FRAMES, prediction_index)
            ret, frame = cap.read()
            if not ret:
                break
            col_tempo = int(prediction_index / np.ceil(frame_rate))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # make predictions here
            cropped_frame, top, bottom = crop_frame(frame)
            masks_list, scores_list = prediction_coral(model, cropped_frame)
            average_score = sum(scores_list) / len(scores_list) if scores_list else 0.0
            
            confidences = [average_score, 1 - average_score]
            
            # Convert to a dictionary with class labels
            confidences_dict = {class_list[i]: round(float(confidences[i]), 3) for i in range(len(class_list))}
            
            # Formatting for CSV output
            predictions_formatted = [f'{confidences_dict[c]:.3f}' for c in class_list]
            
            current_timestamp = initial_timestamp + timedelta(seconds=col_tempo)
            col_timestamp = current_timestamp.strftime('%Y-%m-%d %H:%M:%S')
            
            new_row = [video_name, prediction_index, col_tempo, col_timestamp] + predictions_formatted
            df_inference.append(new_row)

            total_inferences += 1
            
            # Update the progress bar
            pbar.update(1)

    cap.release()
    return df_inference, current_timestamp


def run_inference(videos, columns, save_path):
    #model, class_list = load_model()
    model, class_list = load_model_from_s3(bucket_name, model_key)
    columns.extend(class_list)
    print(columns)
    total_videos = len(videos)
    total_frames = sum([int(cv2.VideoCapture(video).get(cv2.CAP_PROP_FRAME_COUNT)) for video in videos]) // 30
    total_inferences = 0  


    start_total_time = time.time()

    df_inference = []
    init_timestamp = None
    for video in videos:
        result, end_timestamp = process_video(model, video, total_videos, total_inferences, total_frames, class_list, init_timestamp)
        init_timestamp = end_timestamp
        print("init_timestamp: " + str(init_timestamp))
        df_inference.extend(result)
    end_total_time = time.time()
    total_processing_time = end_total_time - start_total_time
    minutes = int(total_processing_time / 60)
    seconds = total_processing_time % 60
    print(f'Total processing time: {minutes} minutes {seconds:.2f} seconds')

    df_inference = pd.DataFrame(df_inference, columns=columns)
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    df_inference.to_csv(os.path.join(save_path, 'predicoes_1s.csv'), index=False)

def upload_videos_to_s3(video_key, video, bucket_name):
    try:
        # Verifique se o vídeo existe no S3
        s3.head_object(Bucket=bucket_name, Key=video_key)
    except botocore.exceptions.ClientError as e:
        if e.response['Error']['Code'] == '404':
            print(f"O vídeo {video_key} não existe no S3")
            return

    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp4") as temp_file:
            s3.download_fileobj(bucket_name, video_key, temp_file)

            video_path = temp_file.name
            s3_video_key = os.path.join(s3_folder_path, "videos", os.path.basename(video))

            content_type, _ = mimetypes.guess_type(video_path)
            extra_args = {'ContentType': content_type} if content_type else {}

            s3.upload_file(video_path, bucket_name, s3_video_key, ExtraArgs=extra_args)

            print(f"Uploaded video {video} to s3://{bucket_name}/{s3_video_key}")

            os.unlink(video_path)  # delete the temp file.
    except Exception as video_error:
        print(f"Error downloading/uploading video: {video_error}")

## Inferência dos vídeos

In [6]:
for video in video_keys:
    videos.append(download_video_from_s3(bucket_name, video))
run_inference(videos, columns, predicoes_path)

['Video', 'Frame', 'Tempo', 'Timestamp', 'CoralSol', 'notCoralSol']
tmplrhzfif_.mp4


Processing tmplrhzfif_: 1861it [01:48, 17.15it/s]                          


init_timestamp: 2025-04-14 14:11:15
tmpyc8h3r84.mp4
{'tmplrhzfif_': datetime.datetime(2025, 4, 14, 13, 41, 17), 'tmpyc8h3r84': datetime.datetime(2025, 4, 14, 14, 11, 15)}


Processing tmpyc8h3r84: 1861it [01:44, 17.85it/s]                          


init_timestamp: 2025-04-14 14:41:13
tmpfaevv0ok.mp4
{'tmplrhzfif_': datetime.datetime(2025, 4, 14, 13, 41, 17), 'tmpyc8h3r84': datetime.datetime(2025, 4, 14, 14, 11, 15), 'tmpfaevv0ok': datetime.datetime(2025, 4, 14, 14, 41, 13)}


...

init_timestamp: 2025-04-14 14:48:35
tmpcskrlxc8.mp4
{'tmplrhzfif_': datetime.datetime(2025, 4, 14, 13, 41, 17), 'tmpyc8h3r84': datetime.datetime(2025, 4, 14, 14, 11, 15), 'tmpfaevv0ok': datetime.datetime(2025, 4, 14, 14, 41, 13), 'tmpcskrlxc8': datetime.datetime(2025, 4, 14, 14, 48, 35)}


Processing tmpcskrlxc8: 1861it [01:41, 18.26it/s]                          


init_timestamp: 2025-04-14 15:18:33
tmp_sa8oqw8.mp4
{'tmplrhzfif_': datetime.datetime(2025, 4, 14, 13, 41, 17), 'tmpyc8h3r84': datetime.datetime(2025, 4, 14, 14, 11, 15), 'tmpfaevv0ok': datetime.datetime(2025, 4, 14, 14, 41, 13), 'tmpcskrlxc8': datetime.datetime(2025, 4, 14, 14, 48, 35), 'tmp_sa8oqw8': datetime.datetime(2025, 4, 14, 15, 18, 33)}


Processing tmp_sa8oqw8: 294it [00:16, 18.36it/s]                         

init_timestamp: 2025-04-14 15:23:16
Total processing time: 5 minutes 56.69 seconds


## Simulando geração dos dados de indexação do YouSub

O arquivo "prediction_index.json" é utilizado para mapear os nomes dos arquivos de predições. Atualmente, ele importa apenas os campos presentes em "predicoes.originais", que deve conter o nome do arquivo CSV, conforme a estrutura estabelecida.

Os campos "suavizada" e "agrupadas" foram projetados para complementar a visualização do gráfico, mas ainda não foram implementados. Portanto, é possível deixá-los vazios, sem afetar o funcionamento atual do sistema.

In [7]:
data = {
    "predicoes": {
        "originais": {
            "periodo": 1,
            "arquivo": "predicoes_1s.csv"
        },
        "suavizadas": {
            "periodo": 1,
            "arquivo": ""
        },
        "agrupadas": {
            "periodo": 1,
            "arquivo": ""
        }
    }
}
json_file_path = os.path.join(result_path, 'prediction_index.json')
with open(json_file_path, 'w') as json_file:
    json.dump(data, json_file, indent=4)
print(f"Arquivo JSON criado em: {json_file_path}")

Arquivo JSON criado em: input_yousub/prediction_index.json


O arquivo pacote.json é gerado com os dados de identificação do pacote, incluindo pacoteHashId e nomePacote. No entanto, apenas o nomePacote é utilizado para alterar o título da página, enquanto o pacoteHashId não é utilizado.

In [8]:
data = {
  "pacoteHashId": "zBLJpJS6zAY",
  "nomePacote": "INSP. PROG. - SIST. ANM PI-ANM 8-RO-52HP-RJS/8-RO-52HPA-RJS / 8-RO-52WA-RJS / P-54"
}

json_file_path = os.path.join(result_path, 'pacote.json')
with open(json_file_path, 'w') as json_file:
    json.dump(data, json_file, indent=4)
print(f"JSON file created at: {json_file_path}")

JSON file created at: input_yousub/pacote.json


Processamento da lista de vídeos e criar um arquivo JSON que contém informações sobre cada vídeo, chamado 'video_data.json'. 
As informações incluem o hash ID do vídeo, nome do arquivo, duração em segundos, caminho do sprite e timestamps inicial e final.

O processo é realizado da seguinte forma:

- O código itera sobre a lista de vídeos e, para cada vídeo, extrai o timestamp inicial, obtém a duração do vídeo e calcula o timestamp final.
- Em seguida, define o caminho do sprite e adiciona as informações do vídeo a uma lista de vídeos.
- O código também gera um sprite para cada vídeo e o salva em um diretório específico.
- Por fim, o código escreve as informações dos vídeos em um arquivo JSON chamado video_data.json, que é criado no diretório especificado.

O arquivo JSON resultante contém uma lista de objetos, cada um representando um vídeo, com as seguintes propriedades: hashid, nome, duracaoSegundos, sprite e timestamps.

In [9]:
video_data = {"videos": []}
index = 1

for video in videos:
    hash_id = index
    initial_timestamp = extract_timestamp_from_filename(os.path.basename(video).split('.')[0], missing_timestamps)
    duration_seconds = get_video_duration(video)
    final_timestamp = calculate_final_timestamp(initial_timestamp, duration_seconds)
    sprite_folder = os.path.join(result_path, 'sprites')
    os.makedirs(sprite_folder, exist_ok=True)
    sprite_path = os.path.join(sprite_folder, f'sprite_{index}.jpg')
    video_info = {
        "hashId": f'hashid_video_{str(hash_id)}',
        "nome": os.path.basename(video),
        "duracaoSegundos": duration_seconds,
        "sprite": f'sprite_{index}.jpg',
        "timestamps": {
            "inicial": str(initial_timestamp),
            "final": str(final_timestamp)
        }
    }
    video_data["videos"].append(video_info)
    miniatures = extract_miniatures(video)
    build_sprite(miniatures, sprite_path)
    index += 1

json_file_path = os.path.join(result_path, 'video_data.json')
with open(json_file_path, 'w') as json_file:
    json.dump(video_data, json_file, indent=4)
print(f"JSON file created at: {json_file_path}")

Final sprite saved at input_yousub/sprites/sprite_1.jpg
Final sprite saved at input_yousub/sprites/sprite_2.jpg
Final sprite saved at input_yousub/sprites/sprite_3.jpg
Final sprite saved at input_yousub/sprites/sprite_4.jpg
Final sprite saved at input_yousub/sprites/sprite_5.jpg
JSON file created at: input_yousub/video_data.json


## Aplicando suavização nas predições, com média móvel (Janela de tamanho 3 e 6)

In [10]:
df = pd.read_csv('input_yousub/predicoes/predicoes_1s.csv')
columns_to_average = df.columns[4:]
window_sizes = [3, 6]
for window_size in window_sizes:
    for column in columns_to_average:
        df[f'{column}_moving_avg_{window_size}'] = df[column].rolling(window=window_size).mean()
output_directory = 'input_yousub/predicoes/smoothed'
os.makedirs(output_directory, exist_ok=True)
output_file_path = os.path.join(output_directory, 'predicoes_1s.csv')
df.to_csv(output_file_path, index=False)

## Enviando pasta input_yousub novamente para o S3

In [11]:
upload_folder_to_s3(local_folder_path, bucket_name, s3_folder_path, video_keys)

for video_key, video in zip(video_keys, videos):
    upload_videos_to_s3(video_key, video, bucket_name)

Uploaded input_yousub/prediction_index.json to s3://analise-dados/projeto-ia-submarina/ia-yousub/NOAA-yousub/output_yousub_NOAA/yousub_input_07-OS006000761716-STRMLZ25-183/prediction_index.json
Uploaded input_yousub/video_data.json to s3://analise-dados/projeto-ia-submarina/ia-yousub/NOAA-yousub/output_yousub_NOAA/yousub_input_07-OS006000761716-STRMLZ25-183/video_data.json
Uploaded input_yousub/pacote.json to s3://analise-dados/projeto-ia-submarina/ia-yousub/NOAA-yousub/output_yousub_NOAA/yousub_input_07-OS006000761716-STRMLZ25-183/pacote.json
Uploaded input_yousub/sprites/sprite_3.jpg to s3://analise-dados/projeto-ia-submarina/ia-yousub/NOAA-yousub/output_yousub_NOAA/yousub_input_07-OS006000761716-STRMLZ25-183/sprites/sprite_3.jpg
Uploaded input_yousub/sprites/sprite_5.jpg to s3://analise-dados/projeto-ia-submarina/ia-yousub/NOAA-yousub/output_yousub_NOAA/yousub_input_07-OS006000761716-STRMLZ25-183/sprites/sprite_5.jpg
Uploaded input_yousub/sprites/sprite_1.jpg to s3://analise-dados/p